# KO/JA clean-damage ablation under `cc_bbox_blur`

**Goal:** cut Clean Δ for KO and JA without giving up much defended accuracy.

The 4-lang trial showed ZH Clean Δ ≈ −1.5pp while KO/JA sit at −11 to −23pp.
ZH is left alone. This notebook only runs `L ∈ {ko, ja}`.

| Attack | Boxes | Score |
|--------|-------|-------|
| `uni_en` | EN + EN | EN + L |
| `uni_l` | L + L | EN + L |
| `multi` | EN + L | EN + L |

**Variants per cell** (Attn-last CAMs cached once, then cheap mask sweeps):

1. `baseline` — thr maximizes tune EN attacked acc; dilate=3, bbox_snap=True
2. `thr_floor_095` — thr fixed at 0.95
3. `pareto_tune` — maximize `en_atk_acc + 0.5 * mean_clean_delta` on tune n=100
4. `tight_dilate` — pareto thr, dilate=1, bbox_snap=True
5. `no_bbox` — pareto thr, dilate=3, bbox_snap=False

Tune n=100 → full n=1000. Geometry: `FONT_SIZE=24`, `NUM_BOXES=2`.


In [ ]:
import importlib.util, subprocess, sys

# Only pip-install missing packages — full resolve can take minutes every run.
_REQUIRED = [
    ('open_clip', 'open_clip_torch'),
    ('transformers', 'transformers'),
    ('datasets', 'datasets'),
    ('matplotlib', 'matplotlib'),
    ('PIL', 'Pillow'),
    ('scipy', 'scipy'),
]
_missing = [pip for mod, pip in _REQUIRED if importlib.util.find_spec(mod) is None]
if _missing:
    print('Installing missing:', _missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=False)
else:
    print('Deps already installed — skipping pip.')


In [ ]:
import os, platform, random, json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from datasets import load_dataset
from transformers import AutoModel, AutoProcessor
from scipy import ndimage

os.makedirs('results', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8
BLUR_RADIUS  = 12
THRESHOLDS   = [0.75, 0.80, 0.85, 0.90, 0.95]
PARETO_THRESHOLDS = [0.85, 0.90, 0.95]
PARTNER_LANGS = ['ko', 'ja']
ATTACKS = ['uni_en', 'uni_l', 'multi']
VARIANTS = ['baseline', 'thr_floor_095', 'pareto_tune', 'tight_dilate', 'no_bbox']

CLASSES = {
    'en': ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck'],
    'ko': ['비행기','자동차','새','고양이','사슴','개','개구리','말','배','트럭'],
    'ja': ['飛行機','自動車','鳥','猫','鹿','犬','カエル','馬','船','トラック'],
}
TMPL = {
    'en': 'a photo of a {}.',
    'ko': '{}의 사진.',
    'ja': '{}の写真。',
}
ALL_LANGS = ['en', 'ko', 'ja']


## 1. Models (EN + KO + JA only)


In [ ]:
def _clip_feat(out):
    if torch.is_tensor(out):
        return out
    if getattr(out, 'pooler_output', None) is not None:
        return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    backend = 'open_clip'
    def __init__(self):
        self.m, _, self.pp = open_clip.create_model_and_transforms(
            'ViT-B-32', pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer('ViT-B-32')
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class KoCLIP:
    lang = 'ko'
    backend = 'hf_vision'
    def __init__(self):
        self.m = AutoModel.from_pretrained(
            'Bingsu/clip-vit-base-patch32-ko',
            attn_implementation='eager').to(DEVICE).eval()
        self.p = AutoProcessor.from_pretrained('Bingsu/clip-vit-base-patch32-ko')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['ko'].format(w) for w in words], padding=True,
                   return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'],
                                       attention_mask=t['attention_mask'])
        return F.normalize(_clip_feat(out), dim=-1)

class JaCLIP:
    lang = 'ja'
    backend = 'open_clip'
    def __init__(self):
        mid = 'hf-hub:llm-jp/llm-jp-clip-vit-base-patch16'
        self.m, _, self.pp = open_clip.create_model_and_transforms(mid)
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer(mid)
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['ja'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

def classify_batch(model, imgs, words, batch_size=128):
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i+batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

MODEL_CLS = {'en': EnCLIP, 'ko': KoCLIP, 'ja': JaCLIP}
models = {}
for lang, cls in MODEL_CLS.items():
    t0 = time.time()
    print(f'Loading {lang}...', end=' ', flush=True)
    models[lang] = cls()
    print(f'{time.time()-t0:.1f}s')
TEXT_EMB = {lang: models[lang].embed_texts(CLASSES[lang]).detach() for lang in ALL_LANGS}
print('Models ready.')


## 2. Dataset + dual-box attack builder


In [ ]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

_sample_path = '../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json'
with open(_sample_path, encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
rows = hf.select(idx)
true = np.array(rows[label_key])
assert len(idx) == 1000 and np.array_equal(true, np.array(_saved['true']))

attack_pos = _saved['attack_pos']
assert len(attack_pos['en']) == len(idx) and len(attack_pos['l']) == len(idx)

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])])
                   for k in range(len(idx))])

clean_224 = [im.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
             for im in rows[image_key]]

all_idx  = np.arange(len(clean_224))
tune_idx = np.concatenate([np.where(true == c)[0][:10] for c in range(10)])
print(f'Loaded {len(clean_224)} images; tune subset = {len(tune_idx)}')
print(f"Attack positions: frozen from sample JSON (ref {attack_pos['ref_bw']}x{attack_pos['ref_bh']})")

_FONT_CACHE = {}

def _font_paths():
    if platform.system() == 'Windows':
        wf = os.path.join(os.environ.get('WINDIR', r'C:\Windows'), 'Fonts')
        cjk = os.path.join(wf, 'msyh.ttc')
        lat = os.path.join(wf, 'arial.ttf')
        ko  = os.path.join(wf, 'malgun.ttf')
        if not os.path.isfile(ko):
            ko = cjk
        return cjk, lat, ko
    cjk = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
    lat = '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
    if not os.path.isfile(cjk):
        cjk = '/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf'
    return cjk, lat, cjk

_CJK_FONT, _LAT_FONT, _KO_FONT = _font_paths()

def _font_for_lang(lang):
    if lang == 'en':
        return _LAT_FONT
    if lang == 'ko':
        return _KO_FONT
    return _CJK_FONT

def _get_font(fp, size=FONT_SIZE):
    key = (fp or '__default__', size)
    if key not in _FONT_CACHE:
        try:
            _FONT_CACHE[key] = ImageFont.truetype(fp, size) if fp else ImageFont.load_default()
        except Exception:
            _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]

def _clamp_xy(xy, bw, bh):
    x, y = int(xy[0]), int(xy[1])
    x = max(0, min(x, max(0, DISPLAY_SIZE - bw)))
    y = max(0, min(y, max(0, DISPLAY_SIZE - bh)))
    return x, y

def draw_dual_box(img, word0, lang0, word1, lang1, img_idx, already_224=False):
    """Place boxes at frozen EN/L anchors from the sample JSON."""
    if not already_224:
        img = img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
    else:
        img = img.copy()
    draw = ImageDraw.Draw(img)
    xy0 = attack_pos['en'][int(img_idx)]
    xy1 = attack_pos['l'][int(img_idx)]
    for word, lang, xy in [(word0, lang0, xy0), (word1, lang1, xy1)]:
        font = _get_font(_font_for_lang(lang))
        bb   = draw.textbbox((0, 0), word, font=font)
        bw   = (bb[2] - bb[0]) + 2 * PAD
        bh   = (bb[3] - bb[1]) + PAD + 12
        rx, ry = _clamp_xy(xy, bw, bh)
        draw.rectangle([rx, ry, rx + bw, ry + bh], fill='white')
        draw.text((rx + PAD - bb[0], ry + PAD - bb[1]), word, fill='black', font=font)
    return img

def build_attack(attack, L):
    out = []
    for i in range(len(clean_224)):
        t = int(target[i])
        en_w = CLASSES['en'][t]
        l_w  = CLASSES[L][t]
        if attack == 'uni_en':
            img = draw_dual_box(clean_224[i], en_w, 'en', en_w, 'en', i, True)
        elif attack == 'uni_l':
            img = draw_dual_box(clean_224[i], l_w, L, l_w, L, i, True)
        else:
            img = draw_dual_box(clean_224[i], en_w, 'en', l_w, L, i, True)
        out.append(img)
    return out

_strip = Image.new('RGB', (DISPLAY_SIZE * 3, 80), (240, 240, 240))
_sd = ImageDraw.Draw(_strip)
for j, (lang, word) in enumerate([
    ('en', 'airplane'), ('ko', '비행기'), ('ja', '飛行機')
]):
    f = _get_font(_font_for_lang(lang), 28)
    _sd.text((j * DISPLAY_SIZE + 10, 20), f'{lang}: {word}', fill='black', font=f)
_strip.save('results/font_check.png')
print('Fonts: LAT=', _LAT_FONT, 'CJK=', _CJK_FONT, 'KO=', _KO_FONT)
print('Saved results/font_check.png')

clean_acc = {
    ml: float((classify_batch(models[ml], clean_224, CLASSES[ml]) == true).mean())
    for ml in ALL_LANGS
}
print('Clean acc:', {k: f'{100*v:.1f}%' for k, v in clean_acc.items()})


## 3. Attn-last saliency (EN/JA open_clip, KO HF vision)


In [ ]:
def _norm_cam(cam):
    cam = np.maximum(cam if isinstance(cam, np.ndarray) else cam.cpu().numpy(), 0)
    cam -= cam.min()
    mx = cam.max()
    return cam / mx if mx > 0 else cam

def align_cam(cam, size=DISPLAY_SIZE):
    return np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    ) / 255.0

def _make_openclip_hook(collector):
    def hook(module, inputs, output):
        q_in = inputs[0]
        if getattr(module, 'batch_first', False):
            B, L, D = q_in.shape
        else:
            L, B, D = q_in.shape
            q_in = q_in.transpose(0, 1).contiguous()
        n_head = module.num_heads
        hd = D // n_head
        with torch.no_grad():
            qkv = F.linear(q_in, module.in_proj_weight, module.in_proj_bias)
            q, k, _ = qkv.chunk(3, dim=-1)
            q = q.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)
            k = k.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)
            attn = (q @ k.transpose(-2, -1)) * (hd ** -0.5)
            attn = attn.softmax(dim=-1)
        collector.append(attn[0].detach().cpu())
    return hook

def _build_attn_cam(all_attns, variant='last'):
    a = all_attns[-1]
    cls_row = a.mean(0)[0, 1:]
    if variant == 'rollout':
        L = all_attns[0].shape[-1]
        rollout = torch.eye(L)
        for att in all_attns:
            a_r = 0.5 * att.mean(0) + 0.5 * torch.eye(L)
            a_r = a_r / a_r.sum(-1, keepdim=True)
            rollout = a_r @ rollout
        cls_row = rollout[0, 1:]
    n = int(round(cls_row.shape[0] ** 0.5))
    return _norm_cam(cls_row.reshape(n, n).numpy())

def classify_and_attn(lang, pil_img, variant='last'):
    # Return (pred, cam) for EN / KO / JA.
    wrapper = models[lang]
    if wrapper.backend == 'open_clip':
        x = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
        collector = []
        handles = [
            rb.attn.register_forward_hook(_make_openclip_hook(collector))
            for rb in wrapper.m.visual.transformer.resblocks
        ]
        with torch.no_grad():
            feat = wrapper.m.visual(x)
            imf = F.normalize(feat, dim=-1)
            pred = int((imf @ TEXT_EMB[lang].T).squeeze().argmax().item())
        for h in handles:
            h.remove()
        return pred, _build_attn_cam(collector, variant)

    pv = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        vis_out = wrapper.m.vision_model(pixel_values=pv, output_attentions=True)
        if hasattr(wrapper.m, 'visual_projection'):
            proj = wrapper.m.visual_projection(vis_out.pooler_output)
        else:
            proj = vis_out.pooler_output
        imf = F.normalize(proj, dim=-1)
        pred = int((imf @ TEXT_EMB[lang].T).squeeze().argmax().item())
    attns = [a[0].cpu() for a in vis_out.attentions]
    return pred, _build_attn_cam(attns, variant)

for lang in ALL_LANGS:
    p, cam = classify_and_attn(lang, clean_224[0], 'last')
    print(f'  {lang}: pred={p} cam={cam.shape}')
print('Saliency ready.')


## 4. Masking helpers (`cc_bbox_blur`, parameterized)


In [ ]:
def n_cam_intersection(*cams):
    return np.minimum.reduce([align_cam(c) for c in cams])

def dilate_mask(mask, iterations=3):
    m = mask.astype(bool)
    for _ in range(iterations):
        pad = np.pad(m, 1, mode='constant', constant_values=False)
        m = (pad[:-2, :-2] | pad[:-2, 1:-1] | pad[:-2, 2:] |
             pad[1:-1, :-2] | pad[1:-1, 1:-1] | pad[1:-1, 2:] |
             pad[2:, :-2] | pad[2:, 1:-1] | pad[2:, 2:])
    return m

def cam_to_mask(saliency, threshold=0.85, dilate=3):
    thr = np.percentile(saliency, threshold * 100)
    mask = saliency >= thr
    if dilate > 0:
        mask = dilate_mask(mask, iterations=dilate)
    return mask

def filter_mask_components(mask, top_k=2, bbox_snap=False):
    labeled, n = ndimage.label(mask.astype(bool))
    if n == 0:
        return mask.astype(bool)
    sizes = [(labeled == i).sum() for i in range(1, n + 1)]
    keep = set(np.argsort(sizes)[::-1][:top_k] + 1)
    out = np.zeros_like(mask, dtype=bool)
    for i in keep:
        comp = labeled == i
        if bbox_snap:
            ys, xs = np.where(comp)
            out[ys.min():ys.max() + 1, xs.min():xs.max() + 1] = True
        else:
            out |= comp
    return out

def apply_mask(pil_img, mask, fill='blur'):
    arr = np.array(pil_img.convert('RGB'))
    m = mask.astype(bool)
    if mask.shape != arr.shape[:2]:
        m = np.array(Image.fromarray(m.astype(np.uint8) * 255).resize(
            arr.shape[1::-1], Image.NEAREST)) > 127
    out = arr.copy()
    if fill == 'blur':
        blurred = np.array(Image.fromarray(arr).filter(
            ImageFilter.GaussianBlur(radius=BLUR_RADIUS)))
        out[m] = blurred[m]
    else:
        mean = arr[~m].mean(0) if (~m).any() else arr.reshape(-1, 3).mean(0)
        out[m] = mean
    return Image.fromarray(out.astype(np.uint8))

def _shrink_to_max_coverage(saliency, mask, max_coverage, dilate, top_k, bbox_snap):
    """Raise percentile until mask coverage <= max_coverage (or thr hits 0.99)."""
    if max_coverage is None or float(mask.mean()) <= max_coverage:
        return mask
    for thr in np.linspace(0.90, 0.99, 10):
        m = cam_to_mask(saliency, thr, dilate=dilate)
        m = filter_mask_components(m, top_k=top_k, bbox_snap=bbox_snap)
        if float(m.mean()) <= max_coverage:
            return m
    return mask

def build_cc_bbox_blur_mask(
    cam_en, cam_l, threshold=0.95, dilate=3, top_k=2,
    bbox_snap=True, max_coverage=None,
):
    inter = n_cam_intersection(cam_en, cam_l)
    mask = cam_to_mask(inter, threshold, dilate=dilate)
    mask = filter_mask_components(mask, top_k=top_k, bbox_snap=bbox_snap)
    if max_coverage is not None and float(mask.mean()) > max_coverage:
        mask = _shrink_to_max_coverage(
            inter, mask, max_coverage, dilate, top_k, bbox_snap)
    return mask

print('Masking helpers ready.')


## 5. CAM cache + defense runner

Attn-last CAMs are computed once per `(image, lang)`, then mask variants reuse them.


In [ ]:
def precompute_cams(L, images, indices, label=''):
    """Return list of (cam_en, cam_l) aligned to indices order."""
    imgs_sub = [images[i] for i in indices]
    n = len(indices)
    print(f'  cams {label} L={L} n={n}...', end=' ', flush=True)
    t0 = time.time()
    cams = []
    for img in imgs_sub:
        _, cam_en = classify_and_attn('en', img, 'last')
        _, cam_l = classify_and_attn(L, img, 'last')
        cams.append((cam_en, cam_l))
    print(f'{time.time() - t0:.1f}s')
    return cams

def run_defense_cached(L, images, indices, cams, mask_cfg, label=''):
    """Apply EN∩L cc_bbox_blur using precomputed cams; return preds for EN and L."""
    imgs_sub = [images[i] for i in indices]
    true_sub = true[indices]
    n = len(indices)
    thr = mask_cfg.get('threshold', 0.95)
    print(f'  defense {label} L={L} n={n} thr={thr} '
          f'dilate={mask_cfg.get("dilate", 3)} '
          f'bbox={mask_cfg.get("bbox_snap", True)}...', end=' ', flush=True)
    t0 = time.time()
    masked_imgs, coverages = [], []
    for img, (cam_en, cam_l) in zip(imgs_sub, cams):
        mask = build_cc_bbox_blur_mask(
            cam_en, cam_l,
            threshold=mask_cfg.get('threshold', 0.95),
            dilate=mask_cfg.get('dilate', 3),
            top_k=mask_cfg.get('top_k', 2),
            bbox_snap=mask_cfg.get('bbox_snap', True),
            max_coverage=mask_cfg.get('max_coverage'),
        )
        coverages.append(float(mask.mean()))
        masked_imgs.append(apply_mask(img, mask, fill='blur'))
    preds = {
        'en': classify_batch(models['en'], masked_imgs, CLASSES['en']),
        L: classify_batch(models[L], masked_imgs, CLASSES[L]),
    }
    elapsed = time.time() - t0
    print(f'{elapsed:.1f}s  cov={100 * np.mean(coverages):.1f}%')
    return {
        'preds': preds,
        'true': true_sub,
        'coverage': float(np.mean(coverages)),
        'time_s': elapsed,
    }

def metrics_for_pair(preds, true_sub, target_sub, L, baseline_acc, baseline_asr):
    out = {}
    for ml in ['en', L]:
        out[ml] = {
            'acc': float((preds[ml] == true_sub).mean()),
            'asr': float((preds[ml] == target_sub).mean()),
            'baseline_acc': baseline_acc[ml],
            'baseline_asr': baseline_asr[ml],
        }
    out['mean_acc'] = 0.5 * (out['en']['acc'] + out[L]['acc'])
    return out

def clean_delta_from_preds(preds, true_sub, L):
    deg = {}
    for ml in ['en', L]:
        masked = float((preds[ml] == true_sub).mean())
        deg[ml] = {
            'baseline_acc': clean_acc[ml],
            'masked_acc': masked,
            'delta_acc': masked - clean_acc[ml],
        }
    mean_d = 0.5 * (deg['en']['delta_acc'] + deg[L]['delta_acc'])
    return deg, mean_d

print('Runner ready.')


## 6. Main loop — variants on cached CAMs

For each `(L, attack)`: build attack → baselines → cache Attn-last cams on
tune + full (attacked and clean) → evaluate five mask variants → pick winner
(best Clean Δ among variants within 3pp mean-acc of baseline).


In [ ]:
def _eval_row(L, attack, variant, mask_cfg, baseline_acc, baseline_asr,
              atk_full, cln_full):
    defense = metrics_for_pair(
        atk_full['preds'], atk_full['true'], target[all_idx], L,
        baseline_acc, baseline_asr)
    clean_deg, mean_clean_delta = clean_delta_from_preds(
        cln_full['preds'], cln_full['true'], L)
    return {
        'L': L,
        'attack': attack,
        'variant': variant,
        'threshold': mask_cfg['threshold'],
        'dilate': mask_cfg.get('dilate', 3),
        'bbox_snap': mask_cfg.get('bbox_snap', True),
        'ran_full': True,
        'clean_acc': {ml: clean_acc[ml] for ml in ['en', L]},
        'baseline_acc': baseline_acc,
        'baseline_asr': baseline_asr,
        'defense': defense,
        'clean_deg': clean_deg,
        'coverage': atk_full['coverage'],
        'mean_acc': defense['mean_acc'],
        'mean_clean_delta': mean_clean_delta,
        'cost': 4,
        'mask_cfg': mask_cfg,
    }


def _pick_winner(variant_rows, baseline_mean_acc):
    """Prefer best (closest to 0) clean Δ among rows within 3pp of baseline mean_acc."""
    eligible = [
        r for r in variant_rows
        if r['mean_acc'] >= baseline_mean_acc - 0.03
    ]
    pool = eligible if eligible else variant_rows
    return max(pool, key=lambda r: r['mean_clean_delta'])


def _tune_baseline_thr(L, attacked, cams_atk_tune):
    best_acc, best_thr, best_cov = -1.0, 0.95, 0.0
    for thr in THRESHOLDS:
        cfg = {'threshold': thr, 'dilate': 3, 'bbox_snap': True}
        res = run_defense_cached(
            L, attacked, tune_idx, cams_atk_tune, cfg, label=f'base thr={thr}')
        en_acc = float((res['preds']['en'] == res['true']).mean())
        l_acc = float((res['preds'][L] == res['true']).mean())
        print(f'    thr={thr} EN={100*en_acc:.1f}% {L.upper()}={100*l_acc:.1f}%')
        if en_acc > best_acc:
            best_acc, best_thr, best_cov = en_acc, thr, res['coverage']
    return best_thr, best_acc, best_cov


def _tune_pareto_thr(L, attacked, cams_atk_tune, cams_cln_tune):
    best_score, best_thr = -1e9, 0.95
    for thr in PARETO_THRESHOLDS:
        cfg = {'threshold': thr, 'dilate': 3, 'bbox_snap': True}
        atk = run_defense_cached(
            L, attacked, tune_idx, cams_atk_tune, cfg, label=f'pareto-atk thr={thr}')
        cln = run_defense_cached(
            L, clean_224, tune_idx, cams_cln_tune, cfg, label=f'pareto-cln thr={thr}')
        en_atk = float((atk['preds']['en'] == atk['true']).mean())
        _, mean_cd = clean_delta_from_preds(cln['preds'], cln['true'], L)
        score = en_atk + 0.5 * mean_cd
        print(f'    pareto thr={thr} en_atk={100*en_atk:.1f}% '
              f'cD={100*mean_cd:+.1f}pp score={score:.3f}')
        if score > best_score:
            best_score, best_thr = score, thr
    return best_thr, best_score


tune_best_cfg = {}
comparison = {}
winners = {}

for L in PARTNER_LANGS:
    for attack in ATTACKS:
        cell_key = f'{L}/{attack}'
        print(f'\n========== {cell_key} ==========')
        out_dir = Path('results') / L / attack
        out_dir.mkdir(parents=True, exist_ok=True)

        attacked = build_attack(attack, L)
        score_langs = ['en', L]

        preds_atk = {
            ml: classify_batch(models[ml], attacked, CLASSES[ml])
            for ml in score_langs
        }
        baseline_acc = {ml: float((preds_atk[ml] == true).mean()) for ml in score_langs}
        baseline_asr = {ml: float((preds_atk[ml] == target).mean()) for ml in score_langs}
        print('  attacked acc:', {k: f'{100*v:.1f}%' for k, v in baseline_acc.items()})
        print('  attacked ASR:', {k: f'{100*v:.1f}%' for k, v in baseline_asr.items()})

        # --- cache CAMs once (tune + full, attacked + clean) ---
        cams_atk_tune = precompute_cams(L, attacked, tune_idx, label='atk-tune')
        cams_cln_tune = precompute_cams(L, clean_224, tune_idx, label='cln-tune')
        cams_atk_full = precompute_cams(L, attacked, all_idx, label='atk-full')
        cams_cln_full = precompute_cams(L, clean_224, all_idx, label='cln-full')

        base_thr, base_tune_en, base_tune_cov = _tune_baseline_thr(
            L, attacked, cams_atk_tune)
        pareto_thr, pareto_score = _tune_pareto_thr(
            L, attacked, cams_atk_tune, cams_cln_tune)

        variant_cfgs = {
            'baseline': {
                'threshold': base_thr, 'dilate': 3, 'bbox_snap': True,
            },
            'thr_floor_095': {
                'threshold': 0.95, 'dilate': 3, 'bbox_snap': True,
            },
            'pareto_tune': {
                'threshold': pareto_thr, 'dilate': 3, 'bbox_snap': True,
            },
            'tight_dilate': {
                'threshold': pareto_thr, 'dilate': 1, 'bbox_snap': True,
            },
            'no_bbox': {
                'threshold': pareto_thr, 'dilate': 3, 'bbox_snap': False,
            },
        }

        # If thr_floor still leaves cov > 12% on tune attacked, enable max_coverage shrink.
        floor_cfg = dict(variant_cfgs['thr_floor_095'])
        floor_tune = run_defense_cached(
            L, attacked, tune_idx, cams_atk_tune, floor_cfg, label='floor-cov-check')
        if floor_tune['coverage'] > 0.12:
            for name in ('thr_floor_095', 'pareto_tune', 'tight_dilate', 'no_bbox'):
                variant_cfgs[name] = dict(variant_cfgs[name])
                variant_cfgs[name]['max_coverage'] = 0.12
            print('  enabled max_coverage=0.12 (floor tune cov '
                  f'{100*floor_tune["coverage"]:.1f}%)')

        tune_best_cfg[cell_key] = {
            'baseline_threshold': base_thr,
            'baseline_tune_en_acc': base_tune_en,
            'baseline_tune_cov': base_tune_cov,
            'pareto_threshold': pareto_thr,
            'pareto_score': pareto_score,
            'L': L,
            'attack': attack,
            'variant_cfgs': variant_cfgs,
        }
        print(f'  baseline thr={base_thr}  pareto thr={pareto_thr}')

        cell_rows = []
        for variant in VARIANTS:
            cfg = variant_cfgs[variant]
            print(f'  --- full {variant} ---')
            atk = run_defense_cached(
                L, attacked, all_idx, cams_atk_full, cfg, label=f'full-atk {variant}')
            cln = run_defense_cached(
                L, clean_224, all_idx, cams_cln_full, cfg, label=f'full-cln {variant}')
            row = _eval_row(
                L, attack, variant, cfg, baseline_acc, baseline_asr, atk, cln)
            print(
                f'  {variant}: mean={100*row["mean_acc"]:.1f}%  '
                f'cD={100*row["mean_clean_delta"]:+.1f}pp  '
                f'cov={100*row["coverage"]:.1f}%'
            )
            with open(out_dir / f'{variant}.json', 'w', encoding='utf-8') as f:
                json.dump(row, f, indent=2)
            comparison[f'{cell_key}/{variant}'] = row
            cell_rows.append(row)

        baseline_row = next(r for r in cell_rows if r['variant'] == 'baseline')
        winner = _pick_winner(cell_rows, baseline_row['mean_acc'])
        winners[cell_key] = {
            'variant': winner['variant'],
            'mean_acc': winner['mean_acc'],
            'mean_clean_delta': winner['mean_clean_delta'],
            'threshold': winner['threshold'],
            'dilate': winner['dilate'],
            'bbox_snap': winner['bbox_snap'],
        }
        print(f'  WINNER {cell_key}: {winner["variant"]}  '
              f'mean={100*winner["mean_acc"]:.1f}%  '
              f'cD={100*winner["mean_clean_delta"]:+.1f}pp')

with open('results/tune_best_cfg.json', 'w', encoding='utf-8') as f:
    json.dump(tune_best_cfg, f, indent=2)
with open('results/comparison_summary.json', 'w', encoding='utf-8') as f:
    json.dump(comparison, f, indent=2)
with open('results/winners.json', 'w', encoding='utf-8') as f:
    json.dump(winners, f, indent=2)
print('\nSaved results/tune_best_cfg.json, comparison_summary.json, winners.json')


## 7. Comparison figure (Clean Δ + mean defended acc by variant)


In [ ]:
variant_colors = {
    'baseline': '#4c72b0',
    'thr_floor_095': '#55a868',
    'pareto_tune': '#c44e52',
    'tight_dilate': '#8172b3',
    'no_bbox': '#ccb974',
}

cells = [(L, a) for L in PARTNER_LANGS for a in ATTACKS]
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

x = np.arange(len(cells))
width = 0.15
offsets = (np.arange(len(VARIANTS)) - (len(VARIANTS) - 1) / 2) * width

# Top: clean damage as positive pp drop (lower bar = less damage = better).
# Bottom: mean defended accuracy under attack (higher = better).
for ax, metric, ylabel, title in [
    (axes[0], 'clean_damage', 'Clean damage (pp)',
     'Clean-image damage (lower = better)'),
    (axes[1], 'mean_acc', 'Mean defended acc (%)',
     'Attacked mean defended accuracy (higher = better)'),
]:
    for vi, variant in enumerate(VARIANTS):
        vals = []
        for L, attack in cells:
            key = f'{L}/{attack}/{variant}'
            row = comparison.get(key)
            if row is None:
                vals.append(0.0)
            elif metric == 'clean_damage':
                vals.append(-100 * row['mean_clean_delta'])  # drop as positive pp
            else:
                vals.append(100 * row['mean_acc'])
        ax.bar(x + offsets[vi], vals, width, label=variant,
               color=variant_colors[variant])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=len(VARIANTS), loc='best')

axes[0].set_ylim(0, None)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{L}/{a}' for L, a in cells], rotation=30, ha='right')
fig.suptitle('KO/JA clean-damage ablation — cc_bbox_blur variants',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig('results/final_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved results/final_comparison.png')

print('\n=== Winners ===')
for k, w in winners.items():
    print(
        f'{k:<12} {w["variant"]:<14} '
        f'mean={100*w["mean_acc"]:5.1f}%  '
        f'cD={100*w["mean_clean_delta"]:+5.1f}pp  '
        f'thr={w["threshold"]:.2f} dilate={w["dilate"]} bbox={w["bbox_snap"]}'
    )

print('\n=== Full table ===')
hdr = (f'{"cell":<28} {"thr":>5} {"dil":>3} {"bbox":>4} '
       f'{"mean":>7} {"cD":>7} {"cov":>6}')
print(hdr)
for k, row in comparison.items():
    print(
        f'{k:<28} {row["threshold"]:5.2f} {row["dilate"]:3d} '
        f'{str(row["bbox_snap"])!s:>4} '
        f'{100*row["mean_acc"]:6.1f}% '
        f'{100*row["mean_clean_delta"]:+6.1f} '
        f'{100*row["coverage"]:5.1f}%'
    )
